# Hybrid Search

キーワード検索（sparse retrieval）とベクトル検索（dense retrieval）を組み合わせる検索方法。実務のRAGでは定番の構成。

## Dense retrievalの弱点

embeddingによるベクトル検索は、言い換えや同義語に強い（「車」と「自動車」がヒットする）一方で、

- 型番・製品名・人名などの固有名詞（「RTX 4090」と「RTX 4080」はベクトル上ほぼ同じ）
- エラーコード、条文番号、IDなどの完全一致が必要な文字列
- embeddingモデルの学習データにない専門用語・社内用語

に弱い。キーワード検索はちょうど逆の性質を持つので、両者を組み合わせると補い合える。

## BM25

キーワード検索の標準的なスコアリング関数。TF-IDFの改良版で、クエリ $Q = (q_1, \dots, q_n)$ に対する文書 $D$ のスコアは

$$
\text{score}(D, Q) = \sum_{i=1}^n \text{IDF}(q_i) \cdot \frac{f(q_i, D) (k_1 + 1)}{f(q_i, D) + k_1 \left(1 - b + b \frac{|D|}{\text{avgdl}} \right)}
$$

- $f(q_i, D)$：文書 $D$ 中の単語 $q_i$ の出現回数
- $|D|$：文書の長さ、$\text{avgdl}$：平均文書長
- $k_1$：単語頻度の飽和の度合い（通常1.2〜2.0）。同じ単語が何度出ても、スコアは一定以上増えない
- $b$：文書長による正規化の強さ（通常0.75）。長い文書が有利になりすぎるのを防ぐ

日本語では単語が空白で区切られていないので、形態素解析（MeCab、Sudachiなど）やn-gramでトークン化してからインデックスする必要がある。トークナイザの選択が検索精度に効く。

## Learned sparse retrieval

BM25の代わりに、ニューラルネットで各単語の重みを学習する手法もある。

- **SPLADE**（[Formal et al., 2021](https://arxiv.org/abs/2107.05720)）：BERTのMLMヘッドを使って語彙全体に対する重みを出力し、文書に出てこない関連語にも重みを付ける（query/document expansion）
- **BGE-M3**（[Chen et al., 2024](https://arxiv.org/abs/2402.03216)）：1つのモデルでdense・sparse・multi-vector（ColBERT型）の3種類の表現を同時に出力できる多言語モデル

転置インデックスが使えるのでBM25と同じ仕組みで高速に検索できる。

## スコアの統合

sparseとdenseは別々に検索して、結果を統合する。両者のスコアはスケールが全く違う（BM25は上限なし、コサイン類似度は-1〜1）ので、そのまま足すことはできない。

### Reciprocal Rank Fusion（RRF）

[Cormack, Clarke, & Buettcher (2009). Reciprocal rank fusion outperforms condorcet and individual rank learning methods.](https://dl.acm.org/doi/10.1145/1571941.1572114)

スコアを捨てて **順位だけ** を使う。複数の検索結果 $R$ について

$$
\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}
$$

を計算し、この値で並べ直す。$k$ は上位の順位を過度に重視しないための定数で、$k=60$ がよく使われる。

スコアの正規化が不要でパラメータもほぼないので、とりあえず使うならこれ。多くのベクトルDB・検索エンジン（Elasticsearch、Weaviate、Qdrantなど）が標準でサポートしている。

### Convex combination

スコアを min-max 正規化などで揃えてから重み付き和をとる。

$$
\text{score}(d) = \alpha \cdot \text{score}_{\text{dense}}(d) + (1 - \alpha) \cdot \text{score}_{\text{sparse}}(d)
$$

[Bruch et al. (2023). An Analysis of Fusion Functions for Hybrid Retrieval](https://arxiv.org/abs/2210.11934) では、少量の評価データで $\alpha$ を調整できるならRRFより良い場合があると報告されている。

## 構成例

1. BM25とベクトル検索でそれぞれtop-50程度を取得
2. RRFで統合
3. [reranker](reranking.ipynb) で並べ直してtop-5〜20をLLMに渡す

Anthropicの [Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) の実験でも、embeddingのみ → +BM25 → +reranking の順に検索失敗率が下がっている（[Chunking](chunking.ipynb) 参照）。